<a href="https://colab.research.google.com/github/qndks11/cv-playground/blob/main/CIFAR10_ResNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pytorch 다운로드

In [9]:
import torch
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
!pip install -q kaggle

# CIFAR 10 다운로드
Toronto server was too slow, so I download from Kaggle instead.

In [12]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from google.colab import files
files.upload()  # upload kaggle.json

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d pankrzysiu/cifar10-python
!unzip cifar10-python.zip -d ./data

Saving kaggle.json to kaggle (2).json
Dataset URL: https://www.kaggle.com/datasets/pankrzysiu/cifar10-python
License(s): other
cifar10-python.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  cifar10-python.zip
replace ./data/cifar-10-batches-py/batches.meta? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace ./data/cifar-10-batches-py/data_batch_1? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace ./data/cifar-10-batches-py/data_batch_2? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace ./data/cifar-10-batches-py/data_batch_3? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace ./data/cifar-10-batches-py/data_batch_4? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace ./data/cifar-10-batches-py/data_batch_5? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace ./data/cifar-10-batches-py/readme.html? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace ./data/cifar-10-batches-py/test_batch? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace ./data/cifar-10-python.ta

In [14]:
# Define a transform to normalize the data
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),                                               # Random Crop Augmentation
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2470, 0.2435, 0.2616])
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2470, 0.2435, 0.2616])
])

# 트레이닝 데이터 다운로드
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
# 테스트 데이터 다운로드
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

# !wget -c https://storage.googleapis.com/tensorflow-datasets-public/cifar-10-python.tar.gz -P ./data

train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size = BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size = BATCH_SIZE, shuffle=True)


# ResNet

https://arxiv.org/pdf/1512.03385

ResNet의 핵심 아이디어는 "identity mapping을 직접 학습하는 대신, residual F(x)을 학습해서 0으로 수렴시키자"는 것이다.

상식적으로 생각했을 때 Layer를 더 쌓으면 성능이 개선되거나 최소한 유지되어야 한다. 예를 들어 4개의 Layer가 있을 때, 레이어를 하나 더 추가하고 그 레이어가 Input을 그대로 Output으로 내보내게 한다면, 성능은 최소한 그대로여야 한다.

하지만 실제로는 Layer가 추가될 때 오히려 성능이 떨어지는 현상이 발생한다. Overfitting 때문도 아니다. Training Loss 자체가 올라가기 때문이다.

원인은 ReLU와 conv 레이어를 쌓아서 identity 함수를 정확히 만들어내는 것이 최적화 관점에서 어렵기 때문이다. 반면 출력을 0으로 만드는 것은 쉽다. 모든 weight를 0으로 두면 되기 때문이다.

그래서 ResNet은 H(x)를 직접 학습하는 대신, H(x) = F(x) + x 형태로 구조를 바꾸고 F(x)를 0으로 수렴시키도록 학습시킨다. F(x)가 0이 되면 자동으로 H(x) = x가 되므로, identity mapping이라는 어려운 목표를 "weight를 0으로 만드는" 쉬운 목표로 바꿔준 것이다.

## BasicBlock

논문에서 나온 B 방법을 사용해서 차원 증가 시에는 x를 stride 2로 1x1 컨볼루션을 해서 높이와 너비는 줄이고 차원은 늘린다.

In [15]:
class BasicBlock(torch.nn.Module):
  def __init__(self, in_channels, out_channels):
    super(BasicBlock, self).__init__()
    self.downsample = in_channels != out_channels # Shortcut을 위한 변수
    stride = 2 if self.downsample else 1
    self.conv1 = torch.nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=3, stride=stride, padding=1) # 스테이지의 첫 번째 basic block만 다름
    self.bn1 = torch.nn.BatchNorm2d(out_channels)
    self.conv2 = torch.nn.Conv2d(in_channels=out_channels, out_channels=out_channels, kernel_size=3, stride=1, padding=1)
    self.bn2 = torch.nn.BatchNorm2d(out_channels) # bn2를 따로 정의하는 이유는 그러지 않을 경우 파라미터가 공유되기 때문.
    self.relu = torch.nn.ReLU(inplace=True)

    if self.downsample:
      self.conv = torch.nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=1, stride=2) # Downsample & Increase dimension

  def forward(self, x):
    org_x = x
    if self.downsample:
      org_x = self.conv(org_x)

    x = self.conv1(x)
    x = self.bn1(x)
    x = self.relu(x)
    x = self.conv2(x)
    x = self.bn2(x)
    x += org_x
    x = self.relu(x)
    return x

In [16]:
def make_layer(in_channels, out_channels, blocks):
  layers = []
  layers.append(BasicBlock(in_channels, out_channels))
  for _ in range(1, blocks):
    layers.append(BasicBlock(out_channels, out_channels))
  return torch.nn.Sequential(*layers)

레이어 개수는 56개이다.

In [17]:
class ResNet(torch.nn.Module):
  def __init__(self):
    super(ResNet, self).__init__()
    self.conv1 = torch.nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
    self.bn = torch.nn.BatchNorm2d(16)
    self.relu = torch.nn.ReLU(inplace=True)
    self.stage1 = make_layer(16, 16, 9)
    self.stage2 = make_layer(16, 32, 9)
    self.stage3 = make_layer(32, 64, 9)
    self.gap = torch.nn.AdaptiveAvgPool2d((1, 1))
    self.fc = torch.nn.Linear(64, 10)

  def forward(self, x):
    x = self.conv1(x)
    x = self.bn(x)
    x = self.relu(x)
    x = self.stage1(x)
    x = self.stage2(x)
    x = self.stage3(x)          # (N, H, W, C)
    x = self.gap(x)             # (N, C, 1, 1)
    x = x.view(x.size(0), -1)   # (N, C)
    x = self.fc(x)              # (N, 10)
    return x

# 훈련
A100 GPU를 쓰는데 그렇게 빠르지 않았다. 사진 크기가 작기 때문에 배치 사이즈를 32에서 512로 늘려 보았다. 그러자 더 빨라졌다. Memory Wall과 Compute Wall을 배웠던 기억이 난다.

정확도가 70%밖에 안 나와서 논문에 나온대로 배치를 128로 바꾸고 Augmentation을 추가했다. 옵티마이저도 Adam에서 SGD로 바꾸고 91번과 136번 epoch에서 1/10으로 작아지게 만들었다. Epoch은 25에서 182로 늘렸다.

In [19]:
model = ResNet().to(device)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=0.0001)   # added weight decay and momentum
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[91, 136], gamma=0.1) # learning rate gets 1/10'ed in 91th and 136th epoch.

BATCH_SIZE = 128
EPOCHS = 182

model.train()

for epoch in range(EPOCHS):
  running_loss = 0.0

  for images, labels in train_loader:
    images, labels = images.to(device), labels.to(device)

    optimizer.zero_grad()              # 그레디언트 초기화
    outputs = model(images)            # Forward
    loss = criterion(outputs, labels)  # loss 계산
    loss.backward()                    # Backward
    optimizer.step()                   # 가중치 업데이트

    running_loss += loss.item()

  scheduler.step()
  avg_loss = running_loss / len(train_loader)
  print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {avg_loss:.4f}")

Epoch [1/182], Loss: 2.1584
Epoch [2/182], Loss: 1.6559
Epoch [3/182], Loss: 1.4343
Epoch [4/182], Loss: 1.2400
Epoch [5/182], Loss: 1.0572
Epoch [6/182], Loss: 0.9043
Epoch [7/182], Loss: 0.7944
Epoch [8/182], Loss: 0.7105
Epoch [9/182], Loss: 0.6504
Epoch [10/182], Loss: 0.5982
Epoch [11/182], Loss: 0.5561
Epoch [12/182], Loss: 0.5251
Epoch [13/182], Loss: 0.5016
Epoch [14/182], Loss: 0.4675
Epoch [15/182], Loss: 0.4475
Epoch [16/182], Loss: 0.4233
Epoch [17/182], Loss: 0.4048
Epoch [18/182], Loss: 0.3952
Epoch [19/182], Loss: 0.3707
Epoch [20/182], Loss: 0.3689
Epoch [21/182], Loss: 0.3451
Epoch [22/182], Loss: 0.3410
Epoch [23/182], Loss: 0.3226
Epoch [24/182], Loss: 0.3137
Epoch [25/182], Loss: 0.3030
Epoch [26/182], Loss: 0.2916
Epoch [27/182], Loss: 0.2826
Epoch [28/182], Loss: 0.2831
Epoch [29/182], Loss: 0.2720
Epoch [30/182], Loss: 0.2621
Epoch [31/182], Loss: 0.2565
Epoch [32/182], Loss: 0.2425
Epoch [33/182], Loss: 0.2397
Epoch [34/182], Loss: 0.2302
Epoch [35/182], Loss: 0

# 평가
정확도 91.25%를 얻었다. 논문에 제시된 93.03%와 얼추 비슷하다.

In [20]:
model.eval()

correct = 0
total = 0

# 틀린 예측 모으기
wrong_images = []
wrong_preds = []
wrong_labels = []

with torch.no_grad():
  for images, labels in test_loader:
    images, labels = images.to(device), labels.to(device)
    outputs = model(images)
    _, predicted = torch.max(outputs, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item() # 맞춘 개수가 나옴
    mask = predicted != labels
    wrong_images.extend(images[mask].cpu())
    wrong_preds.extend(predicted[mask].cpu())
    wrong_labels.extend(labels[mask].cpu())

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 91.25%
